In [1]:
import os
import sys
import time
import warnings
import threading
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

CTABGAN_REPO_DIR = os.path.expanduser("~/CTAB-GAN-Plus")
DATASET_DIR = "./datasets"
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
CTABGAN_EPOCHS_DEFAULT = 150
GENERATE_TIMEOUT_SEC = 600  # hard cap: 10 minutes for generate_samples()

sys.path.insert(0, CTABGAN_REPO_DIR)
_original_cwd = os.getcwd()

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    average_precision_score, balanced_accuracy_score, confusion_matrix,
)
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def get_classifier(name, random_state=RANDOM_STATE):
    if name == "RF":
        return RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=3,
            class_weight="balanced", random_state=random_state, n_jobs=-1,
        )
    elif name == "LGBM":
        return LGBMClassifier(
            n_estimators=100, learning_rate=0.05, num_leaves=31,
            class_weight="balanced", random_state=random_state,
            n_jobs=-1, verbose=-1,
        )
    elif name == "MLP":
        return MLPClassifier(
            hidden_layer_sizes=(128, 64), alpha=0.001,
            max_iter=300, random_state=random_state,
        )


def g_mean_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return float(np.sqrt(sensitivity * specificity))


def evaluate(model, X_test, y_test):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    return {
        "AUC"         : round(roc_auc_score(y_test, y_prob), 4),
        "PR_AUC"      : round(average_precision_score(y_test, y_prob), 4),
        "F1"          : round(f1_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "Precision"   : round(precision_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "Recall"      : round(recall_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "G_mean"      : round(g_mean_score(y_test, y_pred), 4),
        "Balanced_Acc": round(balanced_accuracy_score(y_test, y_pred), 4),
    }


class TimeoutException(Exception):
    pass


def run_with_timeout(func, args=(), kwargs=None, timeout_sec=GENERATE_TIMEOUT_SEC):
    """Runs func in a separate thread; raises TimeoutException if it
    doesn't complete within timeout_sec. Note: the underlying thread
    cannot be forcibly killed in Python, so on timeout the thread keeps
    running in the background, but control returns to the notebook."""
    kwargs = kwargs or {}
    result = {}
    exc = {}

    def target():
        try:
            result["value"] = func(*args, **kwargs)
        except Exception as e:
            exc["error"] = e

    thread = threading.Thread(target=target, daemon=True)
    thread.start()
    thread.join(timeout=timeout_sec)

    if thread.is_alive():
        raise TimeoutException(f"Did not complete within {timeout_sec}s")
    if "error" in exc:
        raise exc["error"]
    return result.get("value")


def apply_ctabgan_plus(X_train, y_train, epochs=CTABGAN_EPOCHS_DEFAULT,
                        random_state=RANDOM_STATE, min_variance=1e-8,
                        max_gen_retries=8):
    """
    Trains CTAB-GAN+ on the FULL training set (majority + minority)
    with the true binary label as problem_type, following the usage
    pattern in the official repository's example notebooks. Synthetic
    minority-class rows (target == 1 after rounding) are filtered from
    the generated output and used to reach full balance
    (Delta = n_maj - n_min), retrying generate_samples() up to
    max_gen_retries times if too few minority rows are produced in a
    single call.

    Zero-variance columns (constant across the full training set) are
    excluded from CTAB-GAN+'s input and reattached post-generation as
    their constant value, to avoid a division-by-zero NaN failure mode
    during CTAB-GAN+'s internal min-max normalisation.
    """
    os.chdir(CTABGAN_REPO_DIR)
    try:
        from model.ctabgan import CTABGAN

        n_maj = int((y_train == 0).sum())
        n_min = int((y_train == 1).sum())
        n_needed = n_maj - n_min

        if n_needed <= 0:
            return X_train, y_train

        n_features = X_train.shape[1]
        col_variance = X_train.var(axis=0)
        constant_col_mask = col_variance < min_variance
        variable_cols_idx = np.where(~constant_col_mask)[0]
        constant_cols_idx = np.where(constant_col_mask)[0]
        constant_values = X_train[0, constant_cols_idx] if len(constant_cols_idx) > 0 else np.array([])

        if len(constant_cols_idx) > 0:
            print(f"    Excluding {len(constant_cols_idx)} zero-variance column(s): "
                  f"indices {constant_cols_idx.tolist()}")

        X_train_variable = X_train[:, variable_cols_idx]
        cols = [f"f{i}" for i in range(X_train_variable.shape[1])]
        df_full = pd.DataFrame(X_train_variable, columns=cols)
        df_full["target"] = y_train

        temp_csv = os.path.join("Real_Datasets", "ctabgan_temp_full.csv")
        os.makedirs("Real_Datasets", exist_ok=True)
        df_full.to_csv(temp_csv, index=False)

        synthesizer = CTABGAN(
            raw_csv_path=temp_csv, test_ratio=0.20,
            categorical_columns=[], log_columns=[], mixed_columns={},
            general_columns=cols, non_categorical_columns=[], integer_columns=[],
            problem_type={"Classification": "target"},
        )
        print("    Fitting (full training set, class-conditional)...")
        synthesizer.fit()

        print(f"    Generating minority-class samples "
              f"(timeout={GENERATE_TIMEOUT_SEC}s/call, max_retries={max_gen_retries})...")
        minority_parts = []
        n_collected = 0
        retry = 0
        while n_collected < n_needed and retry < max_gen_retries:
            synthetic_df = run_with_timeout(synthesizer.generate_samples)
            syn_target_rounded = np.round(synthetic_df["target"].values).astype(int)
            minority_mask = syn_target_rounded == 1

            X_syn_raw = synthetic_df[cols].values.astype(float)
            nan_mask = np.isnan(X_syn_raw).any(axis=1)
            keep_mask = minority_mask & ~nan_mask
            n_kept = int(keep_mask.sum())

            print(f"      Retry {retry}: {len(synthetic_df)} generated, "
                  f"{int(minority_mask.sum())} labeled minority, {n_kept} clean minority kept")

            if n_kept > 0:
                minority_parts.append(X_syn_raw[keep_mask])
                n_collected += n_kept
            retry += 1

        if n_collected == 0:
            raise ValueError("CTAB-GAN+ produced no usable minority-class rows after all retries")

        X_syn_variable = np.vstack(minority_parts)

        if len(X_syn_variable) >= n_needed:
            idx = np.random.RandomState(random_state).choice(len(X_syn_variable), n_needed, replace=False)
        else:
            print(f"      WARNING: only {len(X_syn_variable)}/{n_needed} minority rows "
                  f"available after {max_gen_retries} retries; sampling with replacement")
            idx = np.random.RandomState(random_state).choice(len(X_syn_variable), n_needed, replace=True)
        X_syn_variable = X_syn_variable[idx]

        X_syn = np.zeros((n_needed, n_features))
        X_syn[:, variable_cols_idx] = X_syn_variable
        if len(constant_cols_idx) > 0:
            X_syn[:, constant_cols_idx] = constant_values

        X_maj = X_train[y_train == 0]
        X_min = X_train[y_train == 1]
        X_out = np.vstack([X_maj, X_min, X_syn])
        y_out = np.concatenate([
            np.zeros(len(X_maj)), np.ones(len(X_min)), np.ones(len(X_syn))
        ]).astype(int)
        return X_out, y_out
    finally:
        os.chdir(_original_cwd)


# Confirmed by stability screening: ecoli fails (0% minority yield due to
# its extremely small minority class, n=24, combined with a constant
# column). Excluded from the main run.
EXCLUDED_DATASETS_CTABGAN = {
    "fraud_detection": "projected ~42.3h for 150 epochs (screening model)",
    "protein_homo":    "projected ~21.7h for 150 epochs (screening model)",
    "unsw_nb15":       "projected ~26.0h (screening model); partial run "
                        "confirmed 1 epoch = 7.7 min before interruption",
    "ecoli":           "stability screening confirmed 0% minority-class yield "
                        "(0/235 across generate_samples() calls); minority class "
                        "size (n=24) likely too small for stable conditional "
                        "generation in this implementation",
}
print(f"Final exclusion list: {list(EXCLUDED_DATASETS_CTABGAN.keys())}")
print(f"Datasets to run in main experiment: {16 - len(EXCLUDED_DATASETS_CTABGAN)}")


def run_one_dataset(ds_name, results_accumulator):
    """Runs CTAB-GAN+ + all 3 classifiers for a single dataset, appends
    to results_accumulator (a list), and saves incrementally."""
    path = os.path.join(DATASET_DIR, f"{ds_name}.csv")
    df = pd.read_csv(path)

    bool_cols = df.select_dtypes(include="bool").columns
    if len(bool_cols):
        df[bool_cols] = df[bool_cols].astype("float64")

    X = df.drop(columns=["target"]).values.astype(float)
    y = df["target"].values.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    print(f"{'='*65}")
    print(f"Dataset : {ds_name}  |  n_train={len(X_train):,}  |  minority={y_train.mean():.2%}")
    print(f"{'='*65}")

    t0 = time.time()
    try:
        X_res, y_res = apply_ctabgan_plus(X_train.copy(), y_train.copy())
        gen_time = round(time.time() - t0, 2)
        print(f"  Generated: n={len(y_res):,}  minority={y_res.mean():.2%}  [{gen_time:.1f}s]")
    except TimeoutException as e:
        print(f"  TIMEOUT: {e}")
        return False
    except Exception as e:
        print(f"  CTAB-GAN+ failed: {e}")
        return False

    for clf_name in ["RF", "LGBM", "MLP"]:
        clf = get_classifier(clf_name)
        t1 = time.time()
        try:
            clf.fit(X_res, y_res)
            train_time = round(time.time() - t1, 2)
            metrics = evaluate(clf, X_test, y_test)
        except Exception as e:
            print(f"    {clf_name} - Failed: {e}")
            continue

        results_accumulator.append({
            "dataset": ds_name, "method": "CTAB-GAN+", "classifier": clf_name,
            "generation_time": gen_time, "train_time": train_time,
            "n_synthetic": len(y_res) - len(y_train), **metrics,
        })
        print(f"    {clf_name:<5} | AUC={metrics['AUC']:.4f}  "
              f"PR-AUC={metrics['PR_AUC']:.4f}  G-mean={metrics['G_mean']:.4f}")

    out_path = os.path.join(RESULTS_DIR, "04c_ctabganplus_results.csv")
    pd.DataFrame(results_accumulator).to_csv(out_path, index=False)
    print(f"  === {ds_name} complete; saved to {out_path} ===\n")
    return True


# Shared accumulator across all per-dataset cells below.
# If this is a fresh kernel session, start empty; if resuming, load existing results.
_existing_path = os.path.join(RESULTS_DIR, "04c_ctabganplus_results.csv")
if os.path.exists(_existing_path):
    ctabgan_results = pd.read_csv(_existing_path, keep_default_na=False).to_dict("records")
    print(f"Resumed with {len(ctabgan_results)} existing result rows.")
else:
    ctabgan_results = []
    print("Starting fresh (no existing results file).")

Final exclusion list: ['fraud_detection', 'protein_homo', 'unsw_nb15', 'ecoli']
Datasets to run in main experiment: 12
Resumed with 2 existing result rows.


In [2]:
# Clean up any partial/failed ecoli attempts from prior runs before
# starting the confirmed 12-dataset main experiment.
ctabgan_results = [r for r in ctabgan_results if r["dataset"] != "ecoli"]
print(f"Accumulator after cleanup: {len(ctabgan_results)} rows")
print(f"Datasets present: {sorted(set(r['dataset'] for r in ctabgan_results))}")

Accumulator after cleanup: 0 rows
Datasets present: []


In [3]:
run_one_dataset("pima_diabetes", ctabgan_results)

Dataset : pima_diabetes  |  n_train=537  |  minority=34.82%
    Fitting (full training set, class-conditional)...


100%|██████████| 150/150 [00:17<00:00,  8.57it/s]


Finished training in 19.975010871887207  seconds.
    Generating minority-class samples (timeout=600s/call, max_retries=8)...
      Retry 0: 537 generated, 203 labeled minority, 203 clean minority kept
  Generated: n=700  minority=50.00%  [21.6s]
    RF    | AUC=0.8016  PR-AUC=0.6723  G-mean=0.6948
    LGBM  | AUC=0.7970  PR-AUC=0.6374  G-mean=0.6828
    MLP   | AUC=0.7378  PR-AUC=0.5686  G-mean=0.6453
  === pima_diabetes complete; saved to ./results\04c_ctabganplus_results.csv ===



True

In [4]:
run_one_dataset("yeast_me2", ctabgan_results)

Dataset : yeast_me2  |  n_train=1,038  |  minority=3.47%
    Fitting (full training set, class-conditional)...


100%|██████████| 150/150 [00:17<00:00,  8.56it/s]


Finished training in 17.56199860572815  seconds.
    Generating minority-class samples (timeout=600s/call, max_retries=8)...
      Retry 0: 1038 generated, 268 labeled minority, 268 clean minority kept
      Retry 1: 1038 generated, 240 labeled minority, 240 clean minority kept
      Retry 2: 1038 generated, 258 labeled minority, 258 clean minority kept
      Retry 3: 1038 generated, 249 labeled minority, 249 clean minority kept
  Generated: n=2,004  minority=50.00%  [17.8s]
    RF    | AUC=0.9186  PR-AUC=0.3155  G-mean=0.0000
    LGBM  | AUC=0.8939  PR-AUC=0.3497  G-mean=0.3647
    MLP   | AUC=0.8937  PR-AUC=0.2058  G-mean=0.6199
  === yeast_me2 complete; saved to ./results\04c_ctabganplus_results.csv ===



True

In [5]:
run_one_dataset("ibm_attrition", ctabgan_results)

Dataset : ibm_attrition  |  n_train=1,029  |  minority=16.13%
    Excluding 2 zero-variance column(s): indices [4, 17]
    Fitting (full training set, class-conditional)...


100%|██████████| 150/150 [00:50<00:00,  2.97it/s]


Finished training in 50.59714150428772  seconds.
    Generating minority-class samples (timeout=600s/call, max_retries=8)...
      Retry 0: 1029 generated, 490 labeled minority, 490 clean minority kept
      Retry 1: 1029 generated, 454 labeled minority, 454 clean minority kept
  Generated: n=1,726  minority=50.00%  [50.8s]
    RF    | AUC=0.7656  PR-AUC=0.3913  G-mean=0.3320
    LGBM  | AUC=0.7617  PR-AUC=0.4458  G-mean=0.4521
    MLP   | AUC=0.7381  PR-AUC=0.4493  G-mean=0.6055
  === ibm_attrition complete; saved to ./results\04c_ctabganplus_results.csv ===



True

In [6]:
run_one_dataset("abalone_19", ctabgan_results)

Dataset : abalone_19  |  n_train=2,923  |  minority=0.75%
    Fitting (full training set, class-conditional)...


100%|██████████| 150/150 [01:10<00:00,  2.12it/s]


Finished training in 70.87500071525574  seconds.
    Generating minority-class samples (timeout=600s/call, max_retries=8)...
      Retry 0: 2923 generated, 338 labeled minority, 338 clean minority kept
      Retry 1: 2923 generated, 326 labeled minority, 326 clean minority kept
      Retry 2: 2923 generated, 333 labeled minority, 333 clean minority kept
      Retry 3: 2923 generated, 328 labeled minority, 328 clean minority kept
      Retry 4: 2923 generated, 343 labeled minority, 343 clean minority kept
      Retry 5: 2923 generated, 328 labeled minority, 328 clean minority kept
      Retry 6: 2923 generated, 357 labeled minority, 357 clean minority kept
      Retry 7: 2923 generated, 346 labeled minority, 346 clean minority kept
  Generated: n=5,802  minority=50.00%  [71.7s]
    RF    | AUC=0.7589  PR-AUC=0.0541  G-mean=0.0000
    LGBM  | AUC=0.6661  PR-AUC=0.0249  G-mean=0.0000
    MLP   | AUC=0.6979  PR-AUC=0.0279  G-mean=0.4407
  === abalone_19 complete; saved to ./results\04c_cta

True

In [7]:
run_one_dataset("wine_quality", ctabgan_results)

Dataset : wine_quality  |  n_train=3,428  |  minority=3.73%
    Fitting (full training set, class-conditional)...


100%|██████████| 150/150 [01:29<00:00,  1.69it/s]


Finished training in 89.14751386642456  seconds.
    Generating minority-class samples (timeout=600s/call, max_retries=8)...
      Retry 0: 3428 generated, 266 labeled minority, 266 clean minority kept
      Retry 1: 3428 generated, 252 labeled minority, 252 clean minority kept
      Retry 2: 3428 generated, 262 labeled minority, 262 clean minority kept
      Retry 3: 3428 generated, 281 labeled minority, 281 clean minority kept
      Retry 4: 3428 generated, 274 labeled minority, 274 clean minority kept
      Retry 5: 3428 generated, 249 labeled minority, 249 clean minority kept
      Retry 6: 3428 generated, 269 labeled minority, 269 clean minority kept
      Retry 7: 3428 generated, 270 labeled minority, 270 clean minority kept
  Generated: n=6,600  minority=50.00%  [90.3s]
    RF    | AUC=0.7445  PR-AUC=0.1102  G-mean=0.4864
    LGBM  | AUC=0.7998  PR-AUC=0.1376  G-mean=0.5088
    MLP   | AUC=0.8184  PR-AUC=0.2027  G-mean=0.6795
  === wine_quality complete; saved to ./results\04c_c

True

In [8]:
run_one_dataset("thyroid_sick", ctabgan_results)

Dataset : thyroid_sick  |  n_train=2,640  |  minority=6.14%
    Excluding 1 zero-variance column(s): indices [46]
    Fitting (full training set, class-conditional)...


100%|██████████| 150/150 [03:21<00:00,  1.34s/it]


Finished training in 201.60150790214539  seconds.
    Generating minority-class samples (timeout=600s/call, max_retries=8)...
      Retry 0: 2640 generated, 23 labeled minority, 23 clean minority kept
      Retry 1: 2640 generated, 21 labeled minority, 21 clean minority kept
      Retry 2: 2640 generated, 12 labeled minority, 12 clean minority kept
      Retry 3: 2640 generated, 28 labeled minority, 28 clean minority kept
      Retry 4: 2640 generated, 21 labeled minority, 21 clean minority kept
      Retry 5: 2640 generated, 24 labeled minority, 24 clean minority kept
      Retry 6: 2640 generated, 20 labeled minority, 20 clean minority kept
      Retry 7: 2640 generated, 22 labeled minority, 22 clean minority kept
  Generated: n=4,956  minority=50.00%  [202.4s]
    RF    | AUC=0.9955  PR-AUC=0.9412  G-mean=0.8975
    LGBM  | AUC=0.9978  PR-AUC=0.9587  G-mean=0.9739
    MLP   | AUC=0.9810  PR-AUC=0.8123  G-mean=0.8549
  === thyroid_sick complete; saved to ./results\04c_ctabganplus_res

True

In [9]:
run_one_dataset("churn", ctabgan_results)

Dataset : churn  |  n_train=3,500  |  minority=14.14%
    Fitting (full training set, class-conditional)...


100%|██████████| 150/150 [04:03<00:00,  1.63s/it]


Finished training in 243.97399759292603  seconds.
    Generating minority-class samples (timeout=600s/call, max_retries=8)...
      Retry 0: 3500 generated, 152 labeled minority, 152 clean minority kept
      Retry 1: 3500 generated, 151 labeled minority, 151 clean minority kept
      Retry 2: 3500 generated, 131 labeled minority, 131 clean minority kept
      Retry 3: 3500 generated, 140 labeled minority, 140 clean minority kept
      Retry 4: 3500 generated, 163 labeled minority, 163 clean minority kept
      Retry 5: 3500 generated, 135 labeled minority, 135 clean minority kept
      Retry 6: 3500 generated, 127 labeled minority, 127 clean minority kept
      Retry 7: 3500 generated, 140 labeled minority, 140 clean minority kept
  Generated: n=6,010  minority=50.00%  [245.2s]
    RF    | AUC=0.9258  PR-AUC=0.8791  G-mean=0.7988
    LGBM  | AUC=0.9273  PR-AUC=0.8899  G-mean=0.8765
    MLP   | AUC=0.8904  PR-AUC=0.7502  G-mean=0.8129
  === churn complete; saved to ./results\04c_ctabga

True

In [10]:
run_one_dataset("pageblocks", ctabgan_results)

Dataset : pageblocks  |  n_train=3,831  |  minority=0.52%
    Fitting (full training set, class-conditional)...


100%|██████████| 150/150 [01:45<00:00,  1.42it/s]


Finished training in 105.74800133705139  seconds.
    Generating minority-class samples (timeout=600s/call, max_retries=8)...
      Retry 0: 3831 generated, 310 labeled minority, 310 clean minority kept
      Retry 1: 3831 generated, 286 labeled minority, 286 clean minority kept
      Retry 2: 3831 generated, 269 labeled minority, 269 clean minority kept
      Retry 3: 3831 generated, 271 labeled minority, 271 clean minority kept
      Retry 4: 3831 generated, 273 labeled minority, 273 clean minority kept
      Retry 5: 3831 generated, 260 labeled minority, 260 clean minority kept
      Retry 6: 3831 generated, 298 labeled minority, 298 clean minority kept
      Retry 7: 3831 generated, 248 labeled minority, 248 clean minority kept
  Generated: n=7,622  minority=50.00%  [107.4s]
    RF    | AUC=0.9953  PR-AUC=0.3403  G-mean=0.9932
    LGBM  | AUC=0.9962  PR-AUC=0.4775  G-mean=0.9948
    MLP   | AUC=0.9971  PR-AUC=0.5908  G-mean=0.9969
  === pageblocks complete; saved to ./results\04c_c

True

In [11]:
run_one_dataset("satellite", ctabgan_results)

Dataset : satellite  |  n_train=4,501  |  minority=9.73%
    Fitting (full training set, class-conditional)...


100%|██████████| 150/150 [05:47<00:00,  2.32s/it]


Finished training in 348.1645152568817  seconds.
    Generating minority-class samples (timeout=600s/call, max_retries=8)...
      Retry 0: 4501 generated, 1173 labeled minority, 1173 clean minority kept
      Retry 1: 4501 generated, 1138 labeled minority, 1138 clean minority kept
      Retry 2: 4501 generated, 1152 labeled minority, 1152 clean minority kept
      Retry 3: 4501 generated, 1167 labeled minority, 1167 clean minority kept
  Generated: n=8,126  minority=50.00%  [349.8s]
    RF    | AUC=0.9448  PR-AUC=0.6773  G-mean=0.8549
    LGBM  | AUC=0.9460  PR-AUC=0.7153  G-mean=0.8282
    MLP   | AUC=0.9128  PR-AUC=0.6589  G-mean=0.7918
  === satellite complete; saved to ./results\04c_ctabganplus_results.csv ===



True

In [12]:
run_one_dataset("secom", ctabgan_results)

Dataset : secom  |  n_train=1,096  |  minority=6.66%
    Fitting (full training set, class-conditional)...


100%|██████████| 150/150 [08:02<00:00,  3.22s/it]


Finished training in 483.22403025627136  seconds.
    Generating minority-class samples (timeout=600s/call, max_retries=8)...
      Retry 0: 1096 generated, 65 labeled minority, 65 clean minority kept
      Retry 1: 1096 generated, 74 labeled minority, 74 clean minority kept
      Retry 2: 1096 generated, 80 labeled minority, 80 clean minority kept
      Retry 3: 1096 generated, 75 labeled minority, 75 clean minority kept
      Retry 4: 1096 generated, 70 labeled minority, 70 clean minority kept
      Retry 5: 1096 generated, 69 labeled minority, 69 clean minority kept
      Retry 6: 1096 generated, 78 labeled minority, 78 clean minority kept
      Retry 7: 1096 generated, 68 labeled minority, 68 clean minority kept
  Generated: n=2,046  minority=50.00%  [485.4s]
    RF    | AUC=0.7672  PR-AUC=0.1947  G-mean=0.0000
    LGBM  | AUC=0.8048  PR-AUC=0.3074  G-mean=0.0000
    MLP   | AUC=0.6727  PR-AUC=0.1597  G-mean=0.2517
  === secom complete; saved to ./results\04c_ctabganplus_results.cs

True

In [13]:
run_one_dataset("mammography", ctabgan_results)

Dataset : mammography  |  n_train=7,828  |  minority=2.32%
    Fitting (full training set, class-conditional)...


100%|██████████| 150/150 [03:30<00:00,  1.40s/it]


Finished training in 210.3730013370514  seconds.
    Generating minority-class samples (timeout=600s/call, max_retries=8)...
      Retry 0: 7828 generated, 190 labeled minority, 190 clean minority kept
      Retry 1: 7828 generated, 220 labeled minority, 220 clean minority kept
      Retry 2: 7828 generated, 221 labeled minority, 221 clean minority kept
      Retry 3: 7828 generated, 194 labeled minority, 194 clean minority kept
      Retry 4: 7828 generated, 183 labeled minority, 183 clean minority kept
      Retry 5: 7828 generated, 196 labeled minority, 196 clean minority kept
      Retry 6: 7828 generated, 201 labeled minority, 201 clean minority kept
      Retry 7: 7828 generated, 191 labeled minority, 191 clean minority kept
  Generated: n=15,292  minority=50.00%  [212.7s]
    RF    | AUC=0.9489  PR-AUC=0.6672  G-mean=0.7887
    LGBM  | AUC=0.9493  PR-AUC=0.6951  G-mean=0.7816
    MLP   | AUC=0.9624  PR-AUC=0.7153  G-mean=0.7971
  === mammography complete; saved to ./results\04c_

True

In [14]:
run_one_dataset("credit_default", ctabgan_results)

Dataset : credit_default  |  n_train=21,000  |  minority=22.12%
    Fitting (full training set, class-conditional)...


100%|██████████| 150/150 [27:06<00:00, 10.84s/it]


Finished training in 1627.1481158733368  seconds.
    Generating minority-class samples (timeout=600s/call, max_retries=8)...
      Retry 0: 21000 generated, 3905 labeled minority, 3905 clean minority kept
      Retry 1: 21000 generated, 3870 labeled minority, 3870 clean minority kept
      Retry 2: 21000 generated, 3776 labeled minority, 3776 clean minority kept
      Retry 3: 21000 generated, 3923 labeled minority, 3923 clean minority kept
  Generated: n=32,710  minority=50.00%  [1631.7s]
    RF    | AUC=0.7705  PR-AUC=0.5486  G-mean=0.5817
    LGBM  | AUC=0.7799  PR-AUC=0.5464  G-mean=0.5762
    MLP   | AUC=0.6821  PR-AUC=0.3769  G-mean=0.5278
  === credit_default complete; saved to ./results\04c_ctabganplus_results.csv ===



True

In [15]:
results_04c = pd.read_csv("./results/04c_ctabganplus_results.csv", keep_default_na=False)
print(f"Total rows: {len(results_04c)}")
print(f"Expected: 12 datasets x 3 classifiers = 36")
print(f"Datasets covered: {sorted(results_04c['dataset'].unique())}")
print()
print(f"AUC range: [{results_04c['AUC'].min():.4f}, {results_04c['AUC'].max():.4f}]")
print(f"Any NaN: {results_04c.isna().sum().sum()}")

Total rows: 36
Expected: 12 datasets x 3 classifiers = 36
Datasets covered: ['abalone_19', 'churn', 'credit_default', 'ibm_attrition', 'mammography', 'pageblocks', 'pima_diabetes', 'satellite', 'secom', 'thyroid_sick', 'wine_quality', 'yeast_me2']

AUC range: [0.6661, 0.9978]
Any NaN: 0
